In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/raw/diamonds.csv")


In [3]:
df.shape

(53940, 10)

In [4]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [5]:
df.tail()

,carat,cut,color,clarity,depth,table,price,x,y,z
53935,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74
53939,0.75,Ideal,D,SI2,62.2,55.0,2757,5.83,5.87,3.64


In [6]:
df.describe()

,carat,depth,table,price,x,y,z
count,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
mean,0.797940,61.749405,57.457184,3932.799722,5.731157,5.734526,3.538734
std,0.474011,1.432621,2.234491,3989.439738,1.121761,1.142135,0.705699
min,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y        53940 non-null  float64
 9   z        53940 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 4.1+ MB


In [8]:
df.columns

Index(['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'price', 'x', 'y',
       'z'],
      dtype='object')

In [9]:
df.duplicated().sum()

np.int64(146)

In [10]:
# Zero values in x, y, z are physically impossible for a diamond's dimensions
(df[['x', 'y', 'z']] == 0).sum()

x     8
y     7
z    20
dtype: int64

In [11]:
# Verify category labels match the exact ordinal orders we'll encode against later
df[['cut', 'color', 'clarity']].apply(lambda c: c.unique())

cut         [Ideal, Premium, Good, Very Good, Fair]
color                         [E, I, J, H, F, G, D]
clarity    [SI2, SI1, VS1, VS2, VVS2, VVS1, I1, IF]
dtype: object

In [12]:
# describe() showed y max = 58.9 and z max = 31.8 -- ~10x their medians, likely data-entry errors
df[(df['y'] > 15) | (df['z'] > 15)]

,carat,cut,color,clarity,depth,table,price,x,y,z
24067,2.00,Premium,H,SI2,58.9,57.0,12210,8.09,58.90,8.06
48410,0.51,Very Good,E,VS1,61.8,54.7,1970,5.12,5.15,31.80
49189,0.51,Ideal,E,VS1,61.8,55.0,2075,5.15,31.80,5.12


In [13]:
# Explicit null check (info() implies zero, confirming directly)
df.isnull().sum()

carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64

---
## Section 2 — Data Cleaning

In [14]:
import sys, logging
sys.path.insert(0, "..")
logging.basicConfig(level=logging.INFO, format="%(message)s")

from src.data.clean import load_raw, clean_dataframe, save_artifacts

In [15]:
df_raw = load_raw()
df_clean, imputation_params = clean_dataframe(df_raw)
save_artifacts(df_clean, imputation_params)

Loaded raw data: shape=(53940, 10)


Dropped 146 exact duplicate rows: 53940 → 53794


  y: corrected 2 decimal-placement error(s)


  z: corrected 1 decimal-placement error(s)


Decimal-placement corrections total: 3


...

  y: imputed 6 value(s) — coef=2.2922, intercept=3.9052, R²=0.9542


  z: imputed 19 value(s) — coef=1.4265, intercept=2.4017, R²=0.9532


Cleaning complete — final shape: (53794, 10) | remaining nulls: 0


Saved cleaned CSV: data\processed\diamonds_clean.csv  shape=(53794, 10)


Saved JSON: data\processed\imputation_params.json


In [16]:
print("Imputation params:", imputation_params)
print("Clean shape:", df_clean.shape)
print("Remaining nulls:\n", df_clean.isnull().sum())
df_clean.head()

Imputation params: {'x': {'coef': 2.3112781011087398, 'intercept': 3.8882125273289545}, 'y': {'coef': 2.2922125423120203, 'intercept': 3.905224912833331}, 'z': {'coef': 1.4265155315704798, 'intercept': 2.401733907477745}}
Clean shape: (53794, 10)
Remaining nulls:
 carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
